# Семантическая оценка: BERTScore для GQA-ru

Exact Match отвечает на вопрос, совпал ли ответ посимвольно после нормализации. Для открытых ответов этого недостаточно: «собака» и «пёс» могут быть одинаково верными по смыслу. Поэтому для GQA-ru дополнительно считаем BERTScore — семантическую близость предсказания и эталонного ответа.

Для MMBench-ru BERTScore не применяется: правильный ответ в этом бенчмарке — буква варианта, поэтому там корректна только Accuracy.

In [1]:
# Запустите один раз, если пакет ещё не установлен.
%pip install -q bert-score

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

# На некоторых сетях загрузчик Xet зависает; обычная загрузка HF надёжнее.
os.environ['HF_HUB_DISABLE_XET'] = '1'

import pandas as pd
from bert_score import score

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS_DIR = ROOT / 'results'
# XLM-RoBERTa-large — многоязычная модель, поддерживающая русский язык.
BERTSCORE_MODEL = 'xlm-roberta-large'
DEVICE = 'cpu'  # Не занимает GPU, пока на ней может обучаться другая модель.
BATCH_SIZE = 8

prediction_files = {
    'B0': RESULTS_DIR / 'baseline_predictions.csv',
    'F1-fast': RESULTS_DIR / 'f1_fast_predictions.csv',
}
for name, path in prediction_files.items():
    assert path.exists(), f'Не найден файл предсказаний {name}: {path}'
print('Модель BERTScore:', BERTSCORE_MODEL)
print('Устройство:', DEVICE)

Модель BERTScore: xlm-roberta-large
Устройство: cpu


## 1. Расчёт по сохранённым ответам

BERTScore возвращает три значения для каждого примера: Precision, Recall и F1. Для сравнения вариантов используем средний F1. Параметр `rescale_with_baseline=False` оставляет исходную шкалу метрики и делает результат воспроизводимым при той же версии модели.

In [3]:
def add_bertscore(frame, model_name):
    gqa = frame.loc[frame['benchmark'].eq('GQA-ru')].copy()
    candidates = gqa['prediction'].fillna('').astype(str).tolist()
    references = gqa['target'].fillna('').astype(str).tolist()
    precision, recall, f1 = score(
        candidates, references, model_type=BERTSCORE_MODEL,
        lang='ru', device=DEVICE, batch_size=BATCH_SIZE,
        rescale_with_baseline=False, verbose=True,
    )
    gqa['bertscore_precision'] = precision.cpu().numpy()
    gqa['bertscore_recall'] = recall.cpu().numpy()
    gqa['bertscore_f1'] = f1.cpu().numpy()
    gqa['model'] = model_name
    return gqa

semantic_rows = []
for model_name, path in prediction_files.items():
    print(f'\nСчитаю BERTScore для {model_name}...')
    semantic_rows.append(add_bertscore(pd.read_csv(path), model_name))

semantic_predictions = pd.concat(semantic_rows, ignore_index=True)
display(semantic_predictions[['model', 'sample_id', 'target', 'prediction', 'bertscore_f1']].head())


Считаю BERTScore для B0...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/15 [00:00<?, ?it/s]

...

  0%|          | 0/16 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/13 [00:00<?, ?it/s]

done in 1.29 seconds, 77.31 sentences/sec


,model,sample_id,target,prediction,bertscore_f1
0,B0,20412523,сверху,Снизу,0.854558
1,B0,201576820,да,Нет,0.896670
2,B0,20984271,черный,оранжевый,0.921350
3,B0,201303485,длинный,короткий,0.846303
4,B0,20151632,пластик,Пластик,0.923400


## 2. Сводная таблица и интерпретация

Рост среднего BERTScore F1 означает, что ответы стали ближе к эталонным по смыслу. Он не заменяет Exact Match: модель может дать семантически похожий, но всё же неверный ответ. Поэтому в отчёте приводим обе метрики.

In [4]:
semantic_metrics = (semantic_predictions.groupby('model')[
    ['bertscore_precision', 'bertscore_recall', 'bertscore_f1']
].mean().mul(100).round(2).sort_values('bertscore_f1', ascending=False))

semantic_predictions.to_csv(RESULTS_DIR / 'semantic_predictions_gqa.csv', index=False)
semantic_metrics.to_csv(RESULTS_DIR / 'semantic_metrics_gqa.csv')
display(semantic_metrics)

if {'B0', 'F1-fast'}.issubset(semantic_metrics.index):
    change = semantic_metrics.loc['F1-fast', 'bertscore_f1'] - semantic_metrics.loc['B0', 'bertscore_f1']
    print(f'Изменение F1-fast относительно B0: {change:+.2f} п.п. BERTScore F1.')
print('Сохранены results/semantic_predictions_gqa.csv и results/semantic_metrics_gqa.csv')

,bertscore_precision,bertscore_recall,bertscore_f1
model,,,
B0,87.949997,87.089996,87.470001
F1-fast,85.370003,84.720001,85.000000


Изменение F1-fast относительно B0: -2.47 п.п. BERTScore F1.
Сохранены results/semantic_predictions_gqa.csv и results/semantic_metrics_gqa.csv
